In [1]:
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from typing import List, Tuple
import pandas as pd
from lstm import LSTM
import json


In [ ]:
# A partir da v8

In [2]:
model_names = [f'v{v}' for v in range(1,12+1)]
modelos = {}
configs = {}
for model_name in model_names:
    with open(f"models/{model_name}/config.json", "r", encoding="utf-8") as f:
        config: dict = json.load(f)
    model = LSTM(
        config['vocab_size'],
        config['lstm_emb_size'],
        config['lstm_num_layers'],
        config['lstm_hidden_size'],
        config['lstm_dropout']
    ).to('cpu')
    try:
        model.load_state_dict(torch.load(f"models/{model_name}/model.pt", map_location='cpu'))
    except Exception as e:
        print(model_name, e)
        continue
    

    tokenizer = Tokenizer.from_file(f"artifacts/bpe_{config['vocab_size']}.json")
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    config['parameters'] = trainable_params
    config.pop('device')
    config.pop('exp_name')
    modelos[model_name] = (model, tokenizer)
    configs[model_name] = config

v12 Error(s) in loading state_dict for LSTM:
	Missing key(s) in state_dict: "lstm.weight_ih_l4", "lstm.weight_hh_l4", "lstm.bias_ih_l4", "lstm.bias_hh_l4", "lstm.weight_ih_l5", "lstm.weight_hh_l5", "lstm.bias_ih_l5", "lstm.bias_hh_l5". 


In [4]:
def predicting(initial_text: str, model, max_len, tok: Tokenizer, eos_penalty_base: float, temperature: float, k=1):
    model.eval()

    enc_text_atual = tok.encode(initial_text).ids
    enc_text_atual = torch.tensor(enc_text_atual, dtype=torch.long)
    enc_text_atual = enc_text_atual
    states = None

    for i in range(max_len):
        logits, states = model(enc_text_atual, states)
        logits = logits[-1, :]

        # penaliza EOS
        logits[tok.token_to_id('[EOS]')] *= eos_penalty_base + (i / max_len)

        # temperatura (ANTES do softmax)
        logits = logits / temperature

        probs = torch.softmax(logits, dim=-1)

        # top-k
        topk_probs, topk_indices = torch.topk(probs, k=k)

        # renormaliza
        topk_probs = topk_probs / topk_probs.sum()

        # sample correto
        next_token = torch.multinomial(topk_probs, 1)
        next_token = topk_indices[next_token]

        if next_token.item() == tok.token_to_id('[EOS]'):
            break

        enc_text_atual = torch.cat([
            enc_text_atual,
            next_token
        ])

    text = tok.decode(enc_text_atual.tolist())

    text = text.replace(" ##", "")
    text = text.replace("##", "")

    return text


In [5]:
initial_texts = [
    "São Paulo é um município",
    "O Brasil é um país",
    "A capital da França é",
    "A água é ",
    "O Sol é",
    "A Terra gira em torno",
    "A fotossíntese é",
    "Um vírus é",
    "A energia elétrica é",
    "A linguagem humana é",
    "A democracia é",
    "O sistema solar é",
    "A Lua",
    "A Revolução Industrial",
    "A inteligência artificial é"
    ]


In [60]:
v_atual = 'v8'

In [61]:
predicting(initial_texts[1], modelos[v_atual][0], configs[v_atual]['max_len'], modelos[v_atual][1],1 , 1, k=1)

'O Brasil é um país que se encontra no município de São Paulo , no Brasil , e é o primeiro bairro a ser um dos principais afluentes'